# 02 · Fine-tune DistilBERT on Banking77

Fine-tunes `distilbert-base-uncased` with the Hugging Face `Trainer` and compares it against the TF-IDF baseline from notebook 01 on the **same test split**.

| Setting | Value |
|---|---|
| Epochs | 3 |
| Learning rate | 5e-5 |
| Batch size | 32 |
| Max sequence length | 64 |

**Google Colab:** set *Runtime → Change runtime type → T4 GPU*, then *Run all*. Training uses fp16 on the GPU.

For the side-by-side table, upload `results/baseline_metrics.json` from notebook 01 into `/content/results/`. If the file is missing, the notebook re-runs the identical baseline (it takes a few minutes on CPU).

The fine-tuned model is saved to `./model`, the directory the FastAPI app loads.

In [ ]:
# Colab already ships torch; this only installs or upgrades what is missing.
%pip install -q "datasets>=2.19" "transformers>=4.41" "accelerate>=0.26" scikit-learn pandas

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import ClassLabel, load_dataset
from huggingface_hub import hf_hub_download
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ---- Config (per the project spec) ----
MODEL_NAME = "distilbert-base-uncased"
EPOCHS = 3
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
MAX_LEN = 64
SEED = 42

# Resolve the project root whether the notebook runs from notebooks/, the repo root or Colab (/content).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
MODEL_DIR = ROOT / "model"
BANKING77_REVISION = "90d4e2ee5521c04fc1488f065b8b083658768c57"  # pinned Hub commit of PolyAI/banking77
BANKING77_CSV = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"

set_seed(SEED)
USE_GPU = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if USE_GPU else "cpu"
print("Device:", DEVICE_NAME)
if not USE_GPU:
    print("WARNING: no GPU detected. Training will be slow. On Colab, select a T4 GPU runtime.")


def load_banking77():
    """Load PolyAI/banking77 as a DatasetDict(train, test) with a 77-way ClassLabel `label` column.

    The Hub repo "PolyAI/banking77" is a legacy loading script, which `datasets>=4` no longer
    executes. We load the exact CSVs that script downloads and take the canonical label order
    from the repo's dataset_infos.json, so label ids match the Hub dataset.
    """
    info_path = hf_hub_download("PolyAI/banking77", "dataset_infos.json", repo_type="dataset",
                                revision=BANKING77_REVISION)
    info = json.loads(Path(info_path).read_text(encoding="utf-8"))
    label_feature = ClassLabel(names=info["default"]["features"]["label"]["names"])
    ds = load_dataset("csv", data_files={"train": BANKING77_CSV + "train.csv", "test": BANKING77_CSV + "test.csv"})
    # Map intent names to ids with the canonical order (str2int raises on unknown names).
    ds = ds.map(lambda b: {"label": label_feature.str2int(b["category"])}, batched=True, remove_columns=["category"])
    return ds.cast_column("label", label_feature)

## 1. Load and tokenize

In [ ]:
ds = load_banking77()
label_names = ds["train"].features["label"].names
id2label = dict(enumerate(label_names))
label2id = {name: i for i, name in id2label.items()}
# Plain pandas copies for non-HF consumers (sklearn, length stats).
train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()
print(ds)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# How many queries exceed MAX_LEN tokens? Those are truncated.
lengths = [len(ids) for ids in tokenizer(train_df["text"].tolist())["input_ids"]]
print(f"Train token lengths: median={int(np.median(lengths))}, max={max(lengths)}, "
      f"share > {MAX_LEN}: {np.mean(np.array(lengths) > MAX_LEN):.2%}")

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = ds.map(tokenize, batched=True, remove_columns=["text"])
# Dynamic padding per batch (never beyond MAX_LEN); multiples of 8 suit fp16 tensor cores.
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8 if USE_GPU else None)

## 2. Fine-tune

We train on the full train split with fixed hyperparameters and no checkpoint selection. The test split is used only for the final evaluation, which keeps the comparison with the baseline fair.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names), id2label=id2label, label2id=label2id
)

args = TrainingArguments(
    output_dir=str(ROOT / "outputs"),   # scratch dir; no checkpoints are written
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=128,
    fp16=USE_GPU,                       # T4 supports fp16 (not bf16)
    save_strategy="no",
    logging_steps=50,
    report_to="none",                   # no wandb / tensorboard prompts on Colab
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    data_collator=collator,
)

train_result = trainer.train()
print(train_result.metrics)

## 3. Evaluate on the test split

In [ ]:
pred_out = trainer.predict(tokenized["test"])
y_true = pred_out.label_ids
y_pred = pred_out.predictions.argmax(axis=-1)

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Test accuracy : {accuracy:.4f}")
print(f"Test macro-F1 : {macro_f1:.4f}")

distilbert_metrics = {
    "model": f"{MODEL_NAME} (fine-tuned)",
    "split": "test",
    "n_test": int(len(y_true)),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "max_len": MAX_LEN,
    "device": DEVICE_NAME,
    "train_runtime_s": float(train_result.metrics["train_runtime"]),
}
(RESULTS_DIR / "distilbert_metrics.json").write_text(json.dumps(distilbert_metrics, indent=2))
print(f"Saved {RESULTS_DIR / 'distilbert_metrics.json'}")

## 4. Side-by-side comparison vs. the baseline

This loads `results/baseline_metrics.json` from notebook 01. If the file is missing, the cell re-runs the baseline with exactly the notebook 01 recipe.

In [ ]:
baseline_path = RESULTS_DIR / "baseline_metrics.json"

if baseline_path.exists():
    baseline_metrics = json.loads(baseline_path.read_text())
    print(f"Loaded baseline metrics from {baseline_path}")
else:
    # Same recipe as notebooks/01_baseline.ipynb, kept identical on purpose.
    print("baseline_metrics.json not found, re-running the baseline (identical to notebook 01)...")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.pipeline import Pipeline

    search = GridSearchCV(
        Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)),
            ("clf", LogisticRegression(max_iter=2000)),
        ]),
        param_grid={"clf__C": [1.0, 10.0, 100.0]},
        scoring="f1_macro",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        n_jobs=2,             # each 77-class LBFGS fit needs a few hundred MB; unbounded parallelism can run out of RAM
        error_score="raise",  # fail loudly instead of silently scoring failed fits as NaN
    )
    search.fit(train_df["text"], train_df["label"])
    base_pred = search.best_estimator_.predict(test_df["text"])
    baseline_metrics = {
        "model": "TF-IDF (word 1-2 grams) + LogisticRegression",
        "split": "test",
        "n_test": int(len(test_df)),
        "accuracy": float(accuracy_score(test_df["label"], base_pred)),
        "macro_f1": float(f1_score(test_df["label"], base_pred, average="macro")),
        "best_C": float(search.best_params_["clf__C"]),
    }
    baseline_path.write_text(json.dumps(baseline_metrics, indent=2))

# Guard against comparing numbers from different test sets.
assert baseline_metrics["n_test"] == distilbert_metrics["n_test"], "baseline and DistilBERT used different test sets"

comparison = pd.DataFrame(
    {
        "TF-IDF + LogReg": [baseline_metrics["accuracy"], baseline_metrics["macro_f1"]],
        "DistilBERT": [distilbert_metrics["accuracy"], distilbert_metrics["macro_f1"]],
    },
    index=["Accuracy", "Macro-F1"],
)
comparison["Δ (DistilBERT − baseline)"] = comparison["DistilBERT"] - comparison["TF-IDF + LogReg"]
comparison.to_csv(RESULTS_DIR / "comparison.csv")
print(f"Banking77 test split (n={distilbert_metrics['n_test']})\n")
print(comparison.round(4).to_string())

## 5. Save the model to `./model`

In [ ]:
trainer.save_model(str(MODEL_DIR))       # weights + config.json (includes id2label)
tokenizer.save_pretrained(str(MODEL_DIR))
print(f"Saved to {MODEL_DIR.resolve()}")
print(sorted(p.name for p in MODEL_DIR.iterdir()))

In [ ]:
# Sanity check: reload from disk and classify a few queries, as the API does.
reloaded_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
reloaded = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
queries = ["I still have not received my new card", "How do I top up by bank transfer?", "hmm"]
with torch.inference_mode():
    enc = reloaded_tok(queries, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    probs = torch.softmax(reloaded(**enc).logits, dim=-1)
for q, p in zip(queries, probs):
    conf, idx = p.max(dim=-1)
    print(f"{conf:.3f}  {reloaded.config.id2label[int(idx)]:<40} <- {q!r}")

In [ ]:
# Colab only: zip the model and results and download them, then unzip at the repo root.
try:
    from google.colab import files
    import shutil
    shutil.make_archive("intentroute_model", "zip", root_dir=ROOT, base_dir="model")
    shutil.make_archive("intentroute_results", "zip", root_dir=ROOT, base_dir="results")
    files.download("intentroute_model.zip")
    files.download("intentroute_results.zip")
except ImportError:
    print("Not running on Colab; model is already at", MODEL_DIR)